# 05 · Reload and compare

**Saved base + saved adapter → fresh predictions**

First test the tiny adapter you trained in notebook 04. Then compare the existing Lightning endpoints. These are two separate experiments.

**Who does what:** Nano coordinates the checks. Python reloads your CPU adapter. Lightning base and specialist answer the live comparison question.

###  adapter reload versus serving

[PEFT loading and saving](https://huggingface.co/docs/peft/main/en/quicktour) and the separate [vLLM LoRA serving guide](https://docs.vllm.ai/en/latest/features/lora/).

finish 04, reload your CPU adapter, and compare its outputs with that adapter disabled. The later comparison requires the separately provisioned Lightning API and `train/adapters/jisr-lightning/`; see [service setup](../README.md#provision-services-before-running-the-course).

**Two experiments:** your saved CPU adapter is reattached locally using `PeftModel.from_pretrained`. The live `specialist` alias uses an existing Lightning adapter. This notebook does not deploy your CPU adapter, start a model server, or upload weights anywhere.

The reload, generation comparison, and live HTTP requests execute in the cells below. [05 reference lesson](agent/lessons/05-attach-and-serve.json) is the fixed agent recipe. A nonzero logit difference proves that attachment changes predictions; it does not establish correctness. Keep the generated answer, expected answer, and endpoint comparison separate.

**NVIDIA model identity:** [Lightning NVFP4 model card](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) and [BF16 training checkpoint](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16). Record the serving checkpoint revision, precision, container image, and adapter hash separately from the `base`/`specialist` aliases. This exercise does not deploy or convert the CPU adapter.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Run the executable cells in order. Edit the visible requests, rules, and experiment parameters to change your manual run. The artifact helper only copies prerequisites and records output paths. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import table, bars, loss_chart, show_skill_library
from course.agent.notebook_support import NotebookRun
from course.common import chat, request, rows, BASE_URL
import os, time

lesson = NotebookRun('05-attach-and-serve')
OUTPUT = lesson.work
LIVE = True


### 1 · Reload your CPU adapter

The helper loads fresh copies from disk, compares logits with the adapter enabled and disabled, and generates an answer. A nonzero change proves an effect—not correctness.

In [ ]:
import torch
torch.set_num_threads(2)
from transformers import AutoModelForCausalLM
from peft import PeftModel
PROFILE = 'cpu'
BUILD = OUTPUT/PROFILE
base = AutoModelForCausalLM.from_pretrained(BUILD/'base',local_files_only=True)
adapted = PeftModel.from_pretrained(base,BUILD/'student-adapter',is_trainable=False)
adapted.eval()
print('Attached YOUR adapter:',BUILD/'student-adapter')

In [ ]:
row = json.loads((OUTPUT/'student-data/validation.jsonl').read_text().splitlines()[0])
prompt = 'Observation: '+row['messages'][1]['content']+'\nAnswer: '
ids = torch.tensor([[b+2 for b in prompt.encode()]])
with torch.no_grad():
    with adapted.disable_adapter():
        base_logits = adapted(ids).logits
    adapted_logits = adapted(ids).logits
    generated = adapted.generate(ids,attention_mask=torch.ones_like(ids),max_new_tokens=100,
                                 do_sample=False,pad_token_id=0,eos_token_id=1)
difference = (base_logits-adapted_logits).abs().mean().item()
assert difference>0
print('Mean logit change after attachment:',difference)
print('Fresh generated answer:',bytes([x-2 for x in generated[0,len(ids[0]):].tolist() if 2<=x<258]).decode(errors='replace'))
print('Expected answer:',row['messages'][-1]['content'])

In [ ]:
print("Mean absolute logit change:", difference)
ids = ids
generated = generated
answer = bytes(x-2 for x in generated[0,len(ids[0]):].tolist() if 2 <= x < 258).decode(errors="replace")
table(["Actual CPU answer", "Expected answer"], [(answer, row["messages"][-1]["content"])])

### 2 · Keep the two adapters straight

| Adapter | Used with |
|---|---|
| Your practice adapter | Tiny CPU model from notebook 03 |
| Existing workshop specialist | Lightning on the HTTP endpoint |

We inspect the deployed adapter’s files. We do not replace or deploy anything.

In [ ]:
adapter = ROOT/'train/adapters/jisr-lightning'
for path in sorted(adapter.iterdir()):
    if path.is_file(): print(path.name, path.stat().st_size, 'bytes')
assert (adapter/'adapter_config.json').is_file()
assert (adapter/'adapter_model.safetensors').is_file()

In [ ]:
folder = ROOT / "train/adapters/jisr-lightning"
table(["File", "Bytes"], [(p.name, p.stat().st_size) for p in sorted(folder.iterdir()) if p.is_file()])

### 3 · Ask both Lightning aliases

The same observation goes to `base` and `specialist`. The rules-only prompt is identical for both. Compare the answers with the expected JSON; one example is not a benchmark.

In [ ]:
import json
from urllib.request import Request, urlopen
from train.scripts.jisr_rules import RULES_PROMPT
sample = rows('train/data/jisr/val.jsonl')[0]
messages = [{'role': 'system', 'content': RULES_PROMPT}, sample['messages'][1]]
endpoint_calls = []
for model_id in ('base', 'specialist'):
    started = time.perf_counter()
    payload = {'model': model_id, 'messages': messages, 'temperature': 0, 'max_tokens': 512,
               'response_format': {'type': 'json_object'}, 'chat_template_kwargs': {'enable_thinking': False}}
    http_request = Request(BASE_URL + '/v1/chat/completions', data=json.dumps(payload).encode(),
                           headers={'Content-Type': 'application/json'})
    with urlopen(http_request, timeout=120) as response:
        result = json.load(response)
    assert result['model'] == model_id and result['choices'][0]['finish_reason'] == 'stop'
    answer = result['choices'][0]['message']['content']
    endpoint_calls.append({'model': model_id, 'answer': answer, 'seconds': time.perf_counter()-started})
    print(model_id, answer)
print('Expected:', sample['messages'][-1]['content'])


In [ ]:
print("Observation:", sample["messages"][1]["content"])
table(["Model", "Actual answer", "Seconds"], [(r["model"], r["answer"], r["seconds"]) for r in endpoint_calls])
display(JSON(json.loads(sample["messages"][-1]["content"]), expanded=True))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
assert difference > 0 and len(endpoint_calls) == 2
manual_result = lesson.finish({'mean_logit_change': difference, 'existing_endpoint_answers': endpoint_calls})
display(JSON(manual_result, expanded=False))

### Optional deployment reference · serve a compatible LoRA adapter with NVIDIA NIM

This Markdown example is **not executed by Run All**. It requires an adapter trained for the exact base model in a supported NIM image. The tiny GPT-2 adapter created here is incompatible with the workshop’s Lightning model. Lightning LoRA support must be checked for the specific NIM image, backend, profile, precision, and hardware; this example does not establish that support.

Sources: NVIDIA’s [NIM 2.1.4 LoRA guide](https://docs.nvidia.com/nim/large-language-models/2.1.4/advanced-use-cases/finetune-lora.html), [profile selection](https://docs.nvidia.com/nim/large-language-models/2.1.4/deployment/model-profiles-and-selection.html), and [support matrix](https://docs.nvidia.com/nim/large-language-models/2.1.4/reference/support-matrix.html). Keep the chosen image and documentation version together.

Place the compatible PEFT files in a named subdirectory:

```text
nim-adapters/
└── workshop-adapter/
    ├── adapter_config.json
    └── adapter_model.safetensors
```

After authenticating to NGC, choose a matching image and copy a LoRA-capable profile ID from its `list-model-profiles` output. Replace both placeholders below. Use a separate port and sufficient GPU capacity for this optional deployment.

```bash
export NIM_IMAGE='REPLACE_WITH_COMPATIBLE_PINNED_NIM_IMAGE'
# NGC_API_KEY must already be set in this terminal.
docker run --rm --gpus all -e NGC_API_KEY "$NIM_IMAGE" list-model-profiles
export NIM_MODEL_PROFILE='REPLACE_WITH_LORA_CAPABLE_PROFILE_ID'
export WORKSHOP_NIM_CACHE="$PWD/nim-cache"
mkdir -p "$WORKSHOP_NIM_CACHE"
# NGC_API_KEY must already be set; do not put credentials in this notebook.
docker run --rm --gpus all \
  -e NGC_API_KEY -e NIM_MODEL_PROFILE \
  -e NIM_SERVED_MODEL_NAME=workshop-base \
  -e NIM_PEFT_SOURCE=/opt/nim/loras \
  -v "$WORKSHOP_NIM_CACHE:/opt/nim/.cache" \
  -v "$PWD/nim-adapters:/opt/nim/loras:ro" \
  -p 127.0.0.1:8003:8000 "$NIM_IMAGE"
```

With `NIM_PEFT_REFRESH_INTERVAL` unset, discovery occurs at startup. For dynamic directory watching, NVIDIA documents setting a positive refresh interval; see the [environment-variable reference](https://docs.nvidia.com/nim/large-language-models/2.1.4/reference/environment-variables.html).

In another terminal, verify discovery before inference:

```bash
curl --fail http://127.0.0.1:8003/v1/models
curl --fail http://127.0.0.1:8003/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"workshop-adapter","messages":[{"role":"user","content":"Describe your task briefly."}],"max_tokens":64}'
```

The request above selects an adapter on a **vLLM-backed NIM**. On **SGLang**, use `"model":"workshop-base:workshop-adapter"`; NVIDIA warns that a bare adapter name can select the base instead. Check returned model identity and compare outputs with the base. Discovery and a successful response do not establish adapter quality. The existing workshop endpoints remain a separately provisioned service.

### Grow the skill library
The library now includes this lesson’s skill, **workshop-reload-adapter**. Open the actual instructions below.

**`inspect-adapter` → `reload-adapter` → `check-reload`**

In [ ]:
show_skill_library('05-attach-and-serve')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-reload-adapter/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-reload-adapter/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `02-training-data`, `03-build-lora`, `04-train-adapter`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('05-attach-and-serve')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-reload-adapter/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-adapter', 'reload-adapter', 'check-reload']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the checks. Python reloads your CPU adapter. Lightning base and specialist answer the live comparison question.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Adapter reload changes logits; base and specialist both answer.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** An adapter can change predictions without making them correct. Next, measure multiple answers.

Exit the TUI with `/exit` when done.